In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO

import torch

from compcats.datasets.fmri import nsd
# from compcats.models import resnet50_imagenet1kv2, resnet50_openaiclip
from compcats.utils import model_utils, seed_utils

import main_analysis as ma

# Set seed
SEED = 42
seed_utils.seed_everything(SEED)

# Directories
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)
NSD_DIR = '/weka/scratch/weka/sinha/alishd/datasets/nsd'
# Directories
GRILLSPECTOR_LOCALIZER_DIR = '/net/vast-storage/scratch/vast/sinha/alishd/datasets/\
localizer_grillspector/fLoc_stimuli'
KANWISHER_LOCALIZER_DIR = '/net/vast-storage/scratch/vast/sinha/alishd/datasets/\
localizer_kanwisher/images'
KONKLE_LOCALIZER_DIR = '/net/vast-storage/scratch/vast/sinha/alishd/datasets/\
localizer_konkle/CompSearch'
COCO_TRAIN_ANNOTATIONS_PATH = '/om/user/alishd/datasets/COCO2017/annotations/instances_train2017.json'
COCO_VAL_ANNOTATIONS_PATH = '/om/user/alishd/datasets/COCO2017/annotations/instances_val2017.json'

# Variables
batch_size = 2048
frois = ['FFA', 'PPA', 'EBA']
train_subjects = [1, 2, 5, 7]
test_subjects = [1, 2, 3, 4, 5, 6, 7, 8]
localizer = 'grillspector'
tthreshold = 7
test_model_name = 'resnet50imagenet1kv2'
train_model_name = 'resnet50openaiclip'

# COCO
coco_animals = ['bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe']
coco_face_labels = coco_animals + ['person']
coco_train = COCO(COCO_TRAIN_ANNOTATIONS_PATH)
coco_val = COCO(COCO_VAL_ANNOTATIONS_PATH)

In [ ]:
# Util functions
def get_image_labels(image_id, coco_obj):
    """
    Get the labels for an image given its ID from the COCO dataset.
    
    Args:
        image_id (int): The ID of the image.
        
    Returns:
        list: A list of labels (category names) associated with the image.
    """
    # Get all annotations for the given image
    annotation_ids = coco_obj.getAnnIds(imgIds=image_id)
    annotations = coco_obj.loadAnns(annotation_ids)
    
    # Get category IDs from annotations
    category_ids = {annotation['category_id'] for annotation in annotations}
    
    # Get category names from category IDs
    category_names = [coco_obj.loadCats(cat_id)[0]['name'] for cat_id in category_ids]
    
    return category_names

def min_max_normalize(array):
    """
    Normalize the input array using min-max normalization.

    Parameters:
    array (numpy.ndarray or torch.Tensor): The input array to normalize.

    Returns:
    numpy.ndarray or torch.Tensor: The normalized array.
    """
    min_val = array.min()
    max_val = array.max()
    normalized_array = (array - min_val) / (max_val - min_val)
    return normalized_array

# Interpreting: NSD10k

In [2]:
froi = 'FFA'
subject = 1

# Get subject data
subject_data, subject_stim_map = nsd.get_betas(
    NSD_DIR, subject, froi=froi, tthreshold=tthreshold, voxels=False, repetition_mean=True
)

model = ma.get_model(test_model_name)
all_models_layer = ma.get_froi_models_layers(froi, froi)
layer = all_models_layer[test_model_name]

In [7]:
model_allnsdstim_activations = model_utils.get_dataset_activations(
    MODELS_DIR, NSD_DIR, model, layer, dataset='nsd', froi=froi, tthreshold=tthreshold, \
    froi_localizer=localizer, froi_localizer_dir=ma.get_localizer_dir(localizer), \
    batch_size=batch_size, device='cpu'
)

In [8]:
model_allnsdstim_activations.shape

torch.Size([73000, 729])